# KOMPOSOS-III Chemistry Demo: Solid-State Battery Cell

This notebook demonstrates the three core capabilities of KOMPOSOS-III:
1. **Single-interface compatibility** -- are two materials compatible?
2. **Multi-domain analysis** -- does a full cell design work across all interfaces?
3. **Synthesis planning** -- how do you make the target material?

No neural networks, no training data. Pure compositional reasoning over 169 materials.

In [ ]:
import sys
sys.path.insert(0, '..')

from battery_bridge import ALL_MATERIALS, validate_interface, score_all, get_material
from cross_bridge import MultiDomainAnalyzer, MultiDomainQuery, MultiDomainComponent
from synthesis_planner import SynthesisPlanner, list_all_targets

## 1. Browse Available Materials

KOMPOSOS has 169 materials across 6 domains. Let's see the battery materials.

In [ ]:
print(f"Battery materials: {len(ALL_MATERIALS)}")
print()
for name, mat in sorted(ALL_MATERIALS.items()):
    print(f"  {name:12s}  {mat.formula:20s}  {mat.material_class.name}")

## 2. Single Interface Compatibility

Check if NMC811 cathode is compatible with LLZO solid electrolyte.
The validator runs 5 independent scorers (ion transport, electrochemical stability,
interface compatibility, mechanical compatibility, degradation penalty) and
produces a weighted total.

In [ ]:
result = validate_interface('NMC811', 'LLZO')
print(f"NMC811 <-> LLZO")
print(f"  Total score: {result.total:.3f}")
print(f"  Viable: {result.viable}")
print()
for k, v in result.to_dict().items():
    if k not in ('total', 'viable'):
        print(f"  {k:30s} {v:.4f}")

In [ ]:
# Compare with a known-good pair: LFP + EC (standard liquid cell)
result2 = validate_interface('LFP', 'EC')
print(f"LFP <-> EC")
print(f"  Total score: {result2.total:.3f}")
print(f"  Viable: {result2.viable}")

## 3. Multi-Domain Analysis

Now test a full solid-state cell design spanning 4 material domains:
- **NMC811** (battery domain) -- cathode
- **LLZO** (battery domain) -- solid electrolyte  
- **PEO** (polymer domain) -- binder/interlayer
- **Cu** (metal domain) -- current collector

The multi-domain analyzer identifies which cross-domain functors apply,
scores each interface, and finds the bottleneck.

In [ ]:
query = MultiDomainQuery(
    name="Solid State Cell",
    components=[
        MultiDomainComponent(name="NMC811", role="cathode"),
        MultiDomainComponent(name="LLZO", role="electrolyte"),
        MultiDomainComponent(name="PEO", role="binder"),
        MultiDomainComponent(name="Cu", role="collector"),
    ],
    electrolyte="LLZO",
)

analyzer = MultiDomainAnalyzer()
analysis = analyzer.analyze(query)

print(f"Query: {analysis.query_name}")
print(f"Domains: {', '.join(analysis.domains_involved)}")
print(f"Overall score: {analysis.overall_score:.3f}")
print(f"Viable: {analysis.viable}")
print()

if analysis.bottleneck:
    bn = analysis.bottleneck
    print(f"Bottleneck: {bn.component_a} <-> {bn.component_b}")
    print(f"  Functor: {bn.functor_used}")
    print(f"  Score: {bn.score:.3f}")
print()

print("Cross-domain scores:")
for s in analysis.cross_domain_scores:
    status = 'OK' if s.compatible else 'FAIL'
    print(f"  {s.component_a:8s} <-> {s.component_b:8s}  {s.functor_used:20s}  {s.score:.3f}  [{status}]")

if analysis.warnings:
    print()
    print("Warnings:")
    for w in analysis.warnings:
        print(f"  - {w}")

## 4. Synthesis Planning

Given that LLZO is the electrolyte, how do we actually *make* it?
The synthesis planner ranks routes by feasibility, cost, time, and safety.

In [ ]:
print(f"Available synthesis targets: {', '.join(list_all_targets())}")

In [ ]:
planner = SynthesisPlanner()
syn = planner.plan_synthesis('LFP', available_equipment=['furnace', 'ball_mill', 'mixer'])

print(f"Target: {syn.target}")
print(f"Routes found: {len(syn.routes)}")
print(f"Estimated cost: ${syn.precursor_cost_usd:.2f}")
print(f"Estimated time: {syn.total_time_hours:.1f} hours")
print(f"Equipment needed: {', '.join(syn.equipment_needed)}")
print()

if syn.best_route:
    best = syn.best_route
    print(f"Best route: {best.route.name}")
    print(f"  Composite score: {best.composite_score:.3f}")
    print(f"  Feasibility:     {best.feasibility_score:.3f}")
    print(f"  Cost:            {best.cost_score:.3f}")
    print(f"  Time:            {best.time_score:.3f}")
    print(f"  Safety:          {best.safety_score:.3f}")
    print()
    print("  Steps:")
    for i, step in enumerate(best.route.steps, 1):
        print(f"    {i}. {step.operation}: {step.inputs} -> {step.output}")
        print(f"       T={step.conditions.temperature_C}C, t={step.conditions.time_hours}h, atm={step.conditions.atmosphere}")

## 5. Visualize Compatibility Scores

Compare compatibility across several cathode-electrolyte pairs.

In [ ]:
try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False
    print("matplotlib not installed -- skipping visualization")
    print("Install with: pip install matplotlib")

In [ ]:
if HAS_MPL:
    pairs = [
        ('LFP', 'EC'),
        ('NMC811', 'EC'),
        ('LFP', 'LLZO'),
        ('NMC811', 'LLZO'),
        ('LCO', 'EC'),
        ('Graphite', 'EC'),
    ]

    labels = [f"{a}+{b}" for a, b in pairs]
    scores = []
    for a, b in pairs:
        r = validate_interface(a, b)
        scores.append(r.to_dict())

    totals = [s['total'] for s in scores]
    colors = ['#2ecc71' if t >= 0.45 else '#e74c3c' for t in totals]

    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.barh(labels, totals, color=colors)
    ax.axvline(x=0.45, color='gray', linestyle='--', label='Viability threshold')
    ax.set_xlabel('Compatibility Score')
    ax.set_title('Battery Interface Compatibility')
    ax.set_xlim(0, 1)
    ax.legend()

    for bar, total in zip(bars, totals):
        ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
                f'{total:.2f}', va='center')

    plt.tight_layout()
    plt.show()

In [ ]:
if HAS_MPL:
    # Detailed score breakdown for NMC811 + LLZO
    r = validate_interface('NMC811', 'LLZO')
    d = r.to_dict()
    components = {k: v for k, v in d.items() if k not in ('total', 'viable')}

    fig, ax = plt.subplots(figsize=(8, 5))
    names = [k.replace('_', ' ').title() for k in components.keys()]
    vals = list(components.values())
    colors = ['#3498db' if v >= 0.5 else '#e67e22' if v >= 0.3 else '#e74c3c' for v in vals]

    ax.barh(names, vals, color=colors)
    ax.set_xlabel('Score (0-1)')
    ax.set_title(f'NMC811 + LLZO Score Breakdown (total={d["total"]:.3f})')
    ax.set_xlim(0, 1)

    for i, v in enumerate(vals):
        ax.text(v + 0.02, i, f'{v:.3f}', va='center')

    plt.tight_layout()
    plt.show()

## 6. API Usage

All of the above is also available via the REST API.

```bash
# Start the server
uvicorn api.main:app --reload

# List all materials
curl http://localhost:8000/api/v1/materials

# Check compatibility
curl -X POST http://localhost:8000/api/v1/compatibility \
  -H 'Content-Type: application/json' \
  -d '{"material_a": "NMC811", "material_b": "LLZO"}'

# Multi-domain query
curl -X POST http://localhost:8000/api/v1/multi-domain \
  -H 'Content-Type: application/json' \
  -d '{"name": "Solid State Cell", "components": [{"name": "NMC811", "role": "cathode"}, {"name": "LLZO", "role": "electrolyte"}]}'

# Synthesis planning
curl -X POST http://localhost:8000/api/v1/synthesis \
  -H 'Content-Type: application/json' \
  -d '{"target": "LFP", "available_equipment": ["furnace", "ball_mill"]}'
```

Interactive docs at: http://localhost:8000/docs